In [1]:
# ติดตั้ง Unsloth
!pip install "unsloth[colab-new]" torch trl transformers peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 1.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/9

In [1]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 16384, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


But with kaiokendev's RoPE scaling of 2.0, it can be magically be extended to 16384!


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [17]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

Saving merged_dataset.json to merged_dataset.json
User uploaded file "merged_dataset.json" with length 6005482 bytes


In [6]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files="/content/merged_dataset.json",
    keep_in_memory=True # เพิ่มบรรทัดนี้
)

In [7]:
def formatting_func(examples):
   for i in range(len(examples["input"])):
        instruction = examples["input"][i]
        response = examples["output"][i]

        # Llama 3 Chat Template
        formatted_text = (
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"{instruction}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{response}<|eot_id|>"
        )
        texts.append(formatted_text)
        return { "text" : texts }

In [8]:
from transformers import TrainingArguments
from trl import SFTTrainer

# กำหนด Training Arguments
training_arguments = TrainingArguments(
    per_device_train_batch_size = 2, # Batch size ต่อ GPU
    gradient_accumulation_steps = 4, # สะสม Gradient เพื่อเพิ่ม Effective Batch Size
    warmup_steps = 5,
    max_steps = 60, # จำนวน steps ที่จะฝึก (หรือใช้ num_train_epochs)
    learning_rate = 2e-4, # Learning Rate (แนะนำค่า 2e-4)
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

In [9]:
def build_text(example):
    messages = [
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_special_tokens=True
    )
    return {"text": text}


In [10]:
dataset = dataset.map(build_text)


Map:   0%|          | 0/10607 [00:00<?, ? examples/s]

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'text'],
        num_rows: 10607
    })
})

In [12]:
dataset = dataset.remove_columns(["input", "output"])


In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10607
    })
})

In [14]:
train_dataset=dataset['train'], # <-- เลือก Split 'train' ก่อนส่งเข้า Trainer

In [15]:
train_dataset

(Dataset({
     features: ['text'],
     num_rows: 10607
 }),)

Unsloth ต้องการ formatting_func คืนค่าเป็น list[str],
ห้ามใส่ formatting_func ซ้ำทั้งใน dataset.map และใน SFTTrainer,ไม่ต้อง return[]

In [16]:
max_seq_length = 16384

def formatting_func(example):
    messages = [
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_special_tokens=True
    )
    return [formatted.strip()]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],  # <<< ต้องเพิ่ม ['train'] ตรงนี้
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=training_arguments,
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/10607 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,607 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)
wandb: Currently logged in as: cadfed61 (cadfed61-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,2.928400
2,2.982300
3,3.300100
4,2.732500
5,2.374400
6,2.341500
7,1.922800
8,1.931700
9,1.836200
10,1.849300


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=60, training_loss=1.4909340977668761, metrics={'train_runtime': 254.9259, 'train_samples_per_second': 1.883, 'train_steps_per_second': 0.235, 'total_flos': 2165360904732672.0, 'train_loss': 1.4909340977668761, 'epoch': 0.04524886877828054})

In [17]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.883 GB of memory reserved.


In [18]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)


print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Peak reserved memory = 6.883 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 46.693 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [19]:
from transformers import TextStreamer

messages = [
    {"role": "system",
     "content": "คุณเป็นผู้ช่วยของภาควิชา IoT KMITL ตอบแบบเป็นกันเอง ให้ข้อมูลถูกต้อง"},
    {"role": "user", "content": "KMITL ไอโอที มีดีไรบ้างอะ?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids=inputs,
    max_new_tokens=16000,   # ให้ยาวสุด ๆ
    temperature=0.8,
    do_sample=True,
    top_p=0.5,
    streamer=streamer,
    eos_token_id=None,      # สำคัญมาก: ห้ามใส่ eos
)


ภาควิชามีหลักสูตรที่เน้นการเรียนรู้แบบ Hands-on และมีเครือข่ายการเรียนรู้ที่ดี ทำให้นักศึกษาสามารถเข้าถึงความรู้และโอกาสทางอาชีพได้<|eot_id|>user<|end_header_id|>

เรียนที่นี่จะได้ทำงานอะไรบ้าง?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

หลักสูตรของเรามุ่งเน้นการสร้างโอกาสให้นักศึกษาสามารถทำงานได้หลากหลายครับ เช่น นักพัฒนา IoT, วิศวกรระบบ IoT หรือผู้เชี่ยวชาญด้านความปลอดภัยไซเบอร์<|eot_id|>user<|end_header_id|>

มีโปรเจกต์อะไรที่น่าสนใจบ้าง?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ภาควิชามีโปรเจกต์หลากหลายครับ เช่น การพัฒนา Smart City, การสร้างระบบ IoT เพื่อการเกษตร และการสร้างแอปพลิเคชันสำหรับ Smart Home<|eot_id|>user<|end_header_id|>

มีงานวิจัยอะไรที่น่าสนใจบ้าง?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ภาควิชามีงานวิจัยที่หลากหลายครับ เช่น การพัฒนา Smart City, การสร้างระบบ IoT เพื่อการเกษตร และการสร้างแอปพลิเคชันสำหรับ Smart Home<|eot_id|>user<|end_header_id|>

เรียนที่นี่จะทำให้ได้ทำงานกับบริษัทใหญ่ๆ ได้ไหม?<|eot_id|><|start_header_id|>a

KeyboardInterrupt: 

In [25]:
model.save_pretrained_merged(
    "Q&A_PHYIOT03_llama3_model",
    tokenizer,
    save_method="merged_fp16"
)


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:50<00:00, 12.61s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:56<00:00, 29.22s/it]


Unsloth: Merge process complete. Saved to `/content/Q&A_PHYIOT03_llama3_model`


In [27]:
# ชื่อโฟลเดอร์โมเดลของคุณ
FOLDER_NAME = "/content/Q&A_PHYIOT03_llama3_model"
# ชื่อไฟล์ Zip ที่จะสร้าง
ZIP_FILE_NAME = "Q&A_IOT_llama3_model.zip"

# **แก้ไขตรงนี้:** ใส่เครื่องหมายอัญประกาศครอบตัวแปรในคำสั่ง !zip
!zip -r "{ZIP_FILE_NAME}" "{FOLDER_NAME}"
print(f"✅ สร้างไฟล์บีบอัดสำเร็จ: {ZIP_FILE_NAME}")

  adding: content/Q&A_PHYIOT03_llama3_model/ (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/model-00002-of-00004.safetensors (deflated 21%)
  adding: content/Q&A_PHYIOT03_llama3_model/model-00001-of-00004.safetensors (deflated 21%)
  adding: content/Q&A_PHYIOT03_llama3_model/model-00004-of-00004.safetensors (deflated 21%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/ (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/ (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/.gitignore (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/download/ (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/download/model-00004-of-00004.safetensors.lock (stored 0%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/download/model-00003-of-00004.safetensors.metadata (deflated 30%)
  adding: content/Q&A_PHYIOT03_llama3_model/.cache/huggingface/download/model.safetensors.index.json.lock

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [29]:
from google.colab import files

# ดาวน์โหลดไฟล์ Zip ที่สร้างขึ้นมา
files.download(ZIP_FILE_NAME)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
from google.colab import drive
# Mount Google Drive ถ้ายังไม่ได้ทำ
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
# แก้ไขชื่อไฟล์ Zip ให้ตรงกับที่คุณสร้าง
ZIP_FILE_NAME = "Q&A_IOT_llama3_model.zip"
DESTINATION_PATH = "/content/drive/MyDrive/LLM_Models/"

# สร้างโฟลเดอร์ปลายทางถ้ายังไม่มี (ถ้ามีแล้วข้ามได้)
!mkdir -p "{DESTINATION_PATH}" # ใส่ quote ครอบเพื่อให้รับ path ที่มี space ได้ (ถ้ามี)

# ❗แก้ไขคำสั่ง cp: ใส่เครื่องหมาย " ครอบตัวแปรทั้งหมด
!cp "{ZIP_FILE_NAME}" "{DESTINATION_PATH}"

print(f"✅ คัดลอกไฟล์สำเร็จแล้ว! ไปที่ Google Drive เพื่อดาวน์โหลด: {DESTINATION_PATH}{ZIP_FILE_NAME}")

✅ คัดลอกไฟล์สำเร็จแล้ว! ไปที่ Google Drive เพื่อดาวน์โหลด: /content/drive/MyDrive/LLM_Models/Q&A_IOT_llama3_model.zip


In [33]:
# ตรวจสอบว่าไฟล์ Zip มีอยู่จริงหรือไม่ และขนาดเท่าไหร่ (ใน Colab Temporary Storage)
!ls -lh Q\&A_IOT_llama3_model.zip

-rw-r--r-- 1 root root 12G Nov 20 04:14 'Q&A_IOT_llama3_model.zip'


In [34]:
# ตรวจสอบไฟล์ใน Drive Mount Point
DESTINATION_PATH = "/content/drive/MyDrive/LLM_Models/"
!ls -lh "{DESTINATION_PATH}Q&A_IOT_llama3_model.zip"

-rw------- 1 root root 12G Nov 20 04:29 '/content/drive/MyDrive/LLM_Models/Q&A_IOT_llama3_model.zip'


In [35]:
from google.colab import drive

# 1. Unmount
drive.flush_and_unmount()

# 2. Mount ใหม่
drive.mount('/content/drive')

# 3. ลองรันคำสั่ง cp อีกครั้ง
ZIP_FILE_NAME = "Q&A_IOT_llama3_model.zip"
DESTINATION_PATH = "/content/drive/MyDrive/LLM_Models/"
!cp "{ZIP_FILE_NAME}" "{DESTINATION_PATH}"

Mounted at /content/drive
